This project is about a Technical assistant specializing in automotive diagnostics

# 1. Preparación de datos

## 1.1 Generación de muestras para Fine-tunning

In [1]:
#1. Data preparation
import json
import random
import sys


SYSTEM_PROMPT_RAG = (
    "Eres un asistente técnico experto en diagnóstico automotriz. "
    "Se te proporciona información recuperada de una base de conocimiento verificada. "
    "Basa tu respuesta ÚNICAMENTE en esa información. Si la información recuperada "
    "no contiene la respuesta, indica que no tienes datos suficientes en vez de inventar."
)

# Plantillas para generar variantes de la pregunta del usuario (data augmentation).
# Se agrupan por estilo para poder controlar la mezcla si hace falta.
# Placeholders disponibles: {symptoms} (lista completa), {symptom1} (un solo síntoma al azar), {category}, {subcategory}.

USER_TEMPLATES_FORMAL = [
    "Tengo un vehículo con los siguientes síntomas: {symptoms}. ¿Cuál podría ser el problema y cómo lo diagnostico?",
    "El cliente reporta: {symptoms}. Pertenece al sistema {category}. ¿Qué pasos de diagnóstico recomiendas?",
    "¿Cómo diagnostico una falla en {subcategory} si se presentan estos síntomas: {symptoms}?",
    "Se presenta la siguiente falla en un vehículo, dentro del sistema {category}: {symptoms}. Indique el procedimiento de diagnóstico.",
]

# Estilo "ticket de taller u orden de trabajo"
USER_TEMPLATES_TICKET = [
    "Orden de trabajo: vehículo ingresa por {symptoms}. Sistema sospechoso: {category}. ¿Procedimiento a seguir?",
    "Motivo de ingreso del vehículo: {symptoms}. ¿Qué se debe revisar primero?",
]

# Primera persona del técnico, mecánico, tono coloquial
USER_TEMPLATES_TECNICO = [
    "Estoy revisando un carro y tiene {symptoms}. ¿Por dónde empiezo a diagnosticar?",
    "Oye, tengo un caso raro: {symptoms}. ¿A qué sistema le puede apuntar y cómo lo reviso?",
    "Se me presentó un vehículo con {symptom1}. ¿Qué puede estar pasando?",
]

# Preguntas centradas en un solo síntoma
USER_TEMPLATES_SINTOMA_UNICO = [
    "¿Qué significa que un auto tenga {symptom1}?",
    "El auto presenta {symptom1}, ¿qué reviso primero?",
]

USER_TEMPLATES = (
    USER_TEMPLATES_FORMAL
    + USER_TEMPLATES_TICKET
    + USER_TEMPLATES_TECNICO
    + USER_TEMPLATES_SINTOMA_UNICO
)

# Número de variantes (ejemplos) que se generan por cada entrada de la base de datos.
# Con más plantillas disponibles se puede generar más datos
# para obtener mayor volumen y diversidad de datos por cada entrada de la base de datos.
VARIANTES_POR_ENTRADA = len(USER_TEMPLATES)


def formatear_symptoms(symptoms: list[str]) -> str:
    return "; ".join(s.lower() if i > 0 else s for i, s in enumerate(symptoms))


def construir_respuesta_assistant(entry: dict) -> str:
    """Construye la respuesta del asistente a partir de los pasos de diagnóstico."""
    lineas = [
        f"Esto corresponde al sistema **{entry['category']}**, "
        f"específicamente **{entry['subcategory']}**.",
        "",
        "Pasos de diagnóstico recomendados:",
    ]
    for i, paso in enumerate(entry["diagnosis_steps"], start=1):
        resultados = " / ".join(paso["result"])
        lineas.append(f"{i}. {paso['step']}")
        lineas.append(f"   - Posibles resultados: {resultados}")
    return "\n".join(lineas)


def generar_ejemplos(entry: dict, n_variantes: int = VARIANTES_POR_ENTRADA) -> list[dict]:
    """Genera n ejemplos chat (system/user/assistant) para una entrada del JSON."""
    symptoms_txt = formatear_symptoms(entry["symptoms"])
    respuesta = construir_respuesta_assistant(entry)

    plantillas = random.sample(USER_TEMPLATES, k=min(n_variantes, len(USER_TEMPLATES)))
    ejemplos = []
    for plantilla in plantillas:
        # Para plantillas con {symptom1}, se elige un síntoma al azar de la lista
        symptom1 = random.choice(entry["symptoms"])
        user_msg = plantilla.format(
            symptoms=symptoms_txt,
            symptom1=symptom1,
            category=entry["category"],
            subcategory=entry["subcategory"],
        )
        ejemplos.append(
            {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT_RAG},
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": respuesta},
                ]
            }
        )
    return ejemplos

def dividir_entradas(data: list[dict],train_ratio: float = 0.8,val_ratio: float = 0.1,seed: int = 42,) -> tuple[list[dict], list[dict], list[dict]]:
    """
    Divide las ENTRADAS crudas en train/val/test.
    Esto evita data leakage: variantes del mismo caso nunca quedan repartidas
    entre distintos splits.
    """
    entradas = data.copy()
    rng = random.Random(seed)
    rng.shuffle(entradas)

    n = len(entradas)
    n_train = max(1, round(n * train_ratio))
    n_val = max(1 if n - n_train > 1 else 0, round(n * val_ratio))


    n_train = min(n_train, n)
    n_val = min(n_val, n - n_train)

    train = entradas[:n_train]
    val = entradas[n_train : n_train + n_val]
    test = entradas[n_train + n_val :]
    return train, val, test


def escribir_split(entradas: list[dict], output_path: str, n_variantes: int) -> int:
    ejemplos = []
    for entry in entradas:
        ejemplos.extend(generar_ejemplos(entry, n_variantes=n_variantes))
    with open(output_path, "w", encoding="utf-8") as f:
        for ejemplo in ejemplos:
            f.write(json.dumps(ejemplo, ensure_ascii=False) + "\n")
    return len(ejemplos)


def crearDataset(input_path: str, output_dir: str, train_ratio: float = 0.8, val_ratio: float = 0.1, n_variantes: int = VARIANTES_POR_ENTRADA, seed: int = 42):

    with open(file=input_path, mode="r", encoding="utf-8") as f:
        data = json.load(f)

    train, val, test = dividir_entradas(data, train_ratio, val_ratio, seed)

    import os

    os.makedirs(output_dir, exist_ok=True)
    n_train = escribir_split(train, os.path.join(output_dir, "train.jsonl"), n_variantes)
    n_val = escribir_split(val, os.path.join(output_dir, "val.jsonl"), n_variantes)
    n_test = escribir_split(test, os.path.join(output_dir, "test.jsonl"), n_variantes)

    print(f"Entradas crudas: {len(data)} -> train={len(train)}, val={len(val)}, test={len(test)}")
    print(f"Ejemplos generados: train={n_train}, val={n_val}, test={n_test}")
    print(f"Guardado en: {output_dir}/")

directorio="/content"
archivo_origen = "/content/automotive_faults_esp_aktc_obike_et_al.json"
ruta_destino = directorio+"/messages_to"

train_ratio = 0.8
val_ratio = 0.1
n_variantes = VARIANTES_POR_ENTRADA

crearDataset(input_path=archivo_origen, output_dir=ruta_destino, train_ratio=train_ratio, val_ratio=val_ratio, n_variantes=n_variantes)


Entradas crudas: 99 -> train=79, val=10, test=10
Ejemplos generados: train=869, val=110, test=110
Guardado en: /content/messages_to/


## 1.2 Conexión A huggingface

In [2]:
!pip install peft accelerate trl --quiet
!pip install --upgrade transformers

import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_API_TOKEN'))
print("Sesión de Hugging Face iniciada correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 24.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
Sesión de Hugging Face iniciada correctamente.


## 1.3 Indexación de datos para RAG

In [3]:
import json


def construir_texto_embedding(entry: dict) -> str:
    #Texto que se va a vectorizar para la búsqueda. Prioriza los síntomas,
    #porque así es como los usuarios describen el problema.
    sintomas = "; ".join(entry["symptoms"])
    #return f"Síntomas: {sintomas}; Categoría: ({entry['category']})"
    return sintomas


def construir_texto_contexto(entry: dict) -> str:
    #Texto completo que se le da al LLM como contexto recuperado.
    lineas = [
        f"Categoría: {entry['category']}",
        f"Subcategoría: {entry['subcategory']}",
        f"Síntomas asociados: {', '.join(entry['symptoms'])}",
        "Pasos de diagnóstico:",
    ]
    for i, paso in enumerate(entry["diagnosis_steps"], start=1):
        resultados = " / ".join(paso["result"])
        lineas.append(f"  {i}. {paso['step']} (Posibles resultados: {resultados})")
    return "\n".join(lineas)


def construir_chunks(ruta_json_crudo: str) -> list[dict]:
    """Devuelve una lista de chunks: {id, texto_embedding, texto_contexto, metadata}."""
    with open(ruta_json_crudo, "r", encoding="utf-8") as f:
        data = json.load(f)

    chunks = []
    for i, entry in enumerate(data):
        chunks.append({
            "id": f"doc_{i:04d}",
            "texto_embedding": construir_texto_embedding(entry),
            "texto_contexto": construir_texto_contexto(entry),
            "metadata": {
                "category": entry["category"],
                "subcategory": entry["subcategory"],
            },
        })
    return chunks


#Tomar el mismo archivo del procedimiento para generar muestras para fine-tunning
chunks = construir_chunks(archivo_origen)
print(f"{len(chunks)} chunks construidos.\n")
print("--- Ejemplo (chunk 0) ---")
print("texto_embedding:", chunks[0]["texto_embedding"])
print("\ntexto_contexto:\n", chunks[0]["texto_contexto"])
print("\ntexto_contexto:\n", chunks[0])


99 chunks construidos.

--- Ejemplo (chunk 0) ---
texto_embedding: Luz de advertencia del ABS encendida; Pulsación del pedal de freno

texto_contexto:
 Categoría: Sistema ABS
Subcategoría: Módulo de control ABS
Síntomas asociados: Luz de advertencia del ABS encendida, Pulsación del pedal de freno
Pasos de diagnóstico:
  1. Comprobar el fusible del ABS (Posibles resultados: Fundido / Intacto)
  2. Inspeccionar el cableado del módulo ABS (Posibles resultados: Cableado defectuoso / Cableado correcto)

texto_contexto:
 {'id': 'doc_0000', 'texto_embedding': 'Luz de advertencia del ABS encendida; Pulsación del pedal de freno', 'texto_contexto': 'Categoría: Sistema ABS\nSubcategoría: Módulo de control ABS\nSíntomas asociados: Luz de advertencia del ABS encendida, Pulsación del pedal de freno\nPasos de diagnóstico:\n  1. Comprobar el fusible del ABS (Posibles resultados: Fundido / Intacto)\n  2. Inspeccionar el cableado del módulo ABS (Posibles resultados: Cableado defectuoso / Cableado correc

In [4]:
"""
Aumenta train.jsonl / val.jsonl con ejemplos en formato RAG: el mensaje de
usuario incluye un bloque de "contexto recuperado" (el documento correcto +
1-2 distractores de otras categorías, simulando un retriever imperfecto),
y la respuesta del assistant sigue siendo EXACTAMENTE el mismo formato de
siempre (construir_respuesta_assistant).

IMPORTANTE: usar el MISMO SYSTEM_PROMPT y la MISMA plantilla de mensaje de
usuario que generar_respuesta_rag() en inferencia.
"""

import json
import random


def _plantilla_usuario_rag(contexto: str, pregunta: str) -> str:
    # Debe ser IDÉNTICA a la plantilla usada en generar_respuesta_rag (inferencia).
    return f"Con base en esta información: {contexto}\n\nResponde a: {pregunta}"


def generar_ejemplos_rag_para_entrada(
    entry: dict,
    todos_los_chunks: list[dict],
    preguntas_usuario: list[str],
    n_variantes: int = 3,
    n_distractores: int = 2,
    seed: int | None = None,
) -> list[dict]:
    """
    entry: la entrada cruda (dict con category/subcategory/symptoms/diagnosis_steps)
    todos_los_chunks: chunks de TODO el dataset crudo (para sacar distractores)
    preguntas_usuario: lista de preguntas ya generadas para esta entrada (de
        generar_ejemplos original), para no reinventar plantillas de fraseo
    """
    rng = random.Random(seed)

    chunk_correcto = next(
        c for c in todos_los_chunks
        if c["metadata"]["category"] == entry["category"]
        and c["metadata"]["subcategory"] == entry["subcategory"]
    )
    candidatos_distractores = [
        c for c in todos_los_chunks
        if not (c["metadata"]["category"] == entry["category"]
                and c["metadata"]["subcategory"] == entry["subcategory"])
    ]

    respuesta = construir_respuesta_assistant(entry)
    ejemplos = []

    preguntas_muestreadas = rng.sample(
        preguntas_usuario, k=min(n_variantes, len(preguntas_usuario))
    )
    for pregunta in preguntas_muestreadas:
        distractores = rng.sample(candidatos_distractores, k=min(n_distractores, len(candidatos_distractores)))
        bloques = [chunk_correcto["texto_contexto"]] + [d["texto_contexto"] for d in distractores]
        rng.shuffle(bloques)  # el documento correcto no siempre va primero
        contexto = "\n\n---\n\n".join(bloques)

        user_msg = _plantilla_usuario_rag(contexto, pregunta)
        ejemplos.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT_RAG},
                {"role": "user", "content": user_msg},
                {"role": "assistant", "content": respuesta},
            ]
        })
    return ejemplos


def aumentar_split_con_rag(
    ruta_jsonl_original: str,
    ruta_json_crudo: str,
    ruta_jsonl_salida: str,
    n_variantes_rag: int = 3,
    n_distractores: int = 2,
    seed: int = 42,
):
    """Lee un split ya generado (train.jsonl/val.jsonl), agrega ejemplos RAG
    para las mismas entradas, y escribe un nuevo archivo con AMBOS tipos
    mezclados (originales sin contexto + nuevos con contexto)."""
    todos_los_chunks = construir_chunks(ruta_json_crudo)

    # Agrupar las preguntas de usuario ya existentes por (categoria, subcategoria),
    # para reutilizar el mismo fraseo variado en los ejemplos RAG.
    ejemplos_originales = []
    preguntas_por_entrada: dict[tuple, list[str]] = {}
    with open(ruta_jsonl_original, encoding="utf-8") as f:
        for linea in f:
            ej = json.loads(linea)
            ejemplos_originales.append(ej)
            user_msg = next(m["content"] for m in ej["messages"] if m["role"] == "user")
            resp = next(m["content"] for m in ej["messages"] if m["role"] == "assistant")
            # Usamos la respuesta para encontrar a qué chunk corresponde esta pregunta
            for c in todos_los_chunks:
                if c["texto_embedding"].split(" - ")[0] in resp and c["metadata"]["subcategory"] in resp:
                    clave = (c["metadata"]["category"], c["metadata"]["subcategory"])
                    preguntas_por_entrada.setdefault(clave, []).append(user_msg)
                    break

    with open(ruta_json_crudo, encoding="utf-8") as f:
        data_cruda = json.load(f)

    rng = random.Random(seed)
    nuevos_ejemplos_rag = []
    for entry in data_cruda:
        clave = (entry["category"], entry["subcategory"])
        preguntas = preguntas_por_entrada.get(clave)
        if not preguntas:
            continue  # esta entrada no está en este split (train vs val vs test)
        nuevos_ejemplos_rag.extend(
            generar_ejemplos_rag_para_entrada(
                entry, todos_los_chunks, preguntas,
                n_variantes=n_variantes_rag, n_distractores=n_distractores,
                seed=rng.randint(0, 1_000_000),
            )
        )

    todos = ejemplos_originales + nuevos_ejemplos_rag
    rng.shuffle(todos)

    with open(ruta_jsonl_salida, "w", encoding="utf-8") as f:
        for ej in todos:
            f.write(json.dumps(ej, ensure_ascii=False) + "\n")

    print(f"{ruta_jsonl_salida}: {len(ejemplos_originales)} originales + "
          f"{len(nuevos_ejemplos_rag)} con contexto RAG = {len(todos)} totales")




aumentar_split_con_rag(
    f"{ruta_destino}/train.jsonl", archivo_origen, f"{ruta_destino}/train_rag.jsonl",
)
aumentar_split_con_rag(
    f"{ruta_destino}/val.jsonl", archivo_origen, f"{ruta_destino}/val_rag.jsonl",
)

/content/messages_to/train_rag.jsonl: 869 originales + 0 con contexto RAG = 869 totales
/content/messages_to/val_rag.jsonl: 110 originales + 0 con contexto RAG = 110 totales


In [5]:
import pickle

import numpy as np
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
RUTA_EMBEDDINGS = directorio+"/rag_embeddings.npy"
RUTA_CHUNKS = directorio+"/rag_chunks.pkl"

_modelo_embeddings = None
_embeddings_docs = None
_chunks = None

def construir_y_guardar_indice(RUTA_JSON_CRUDO):
    """Genera embeddings normalizados para todos los documentos y los guarda a disco."""
    global _modelo_embeddings, _embeddings_docs, _chunks

    chunks = construir_chunks(RUTA_JSON_CRUDO)
    documentos = [chunk["texto_embedding"] for chunk in chunks]

    modelo_challenge_embeddings = SentenceTransformer(EMBEDDING_MODEL)
    embeddings_docs = modelo_challenge_embeddings.encode(
        documentos,
        normalize_embeddings=True,  # necesario para que el producto punto == similitud coseno
        show_progress_bar=True,
    )
    embeddings_docs = np.array(embeddings_docs, dtype="float32")
    print("Embeddings generados:", embeddings_docs.shape)

    np.save(RUTA_EMBEDDINGS, embeddings_docs)
    with open(RUTA_CHUNKS, "wb") as f:
        pickle.dump(chunks, f)
    print(f"Embeddings guardados en {RUTA_EMBEDDINGS}, chunks en {RUTA_CHUNKS}")

    _modelo_embeddings = modelo_challenge_embeddings
    _embeddings_docs = embeddings_docs
    _chunks = chunks

def _cargar_recursos():
    """Carga los embeddings ya guardados a disco (uso normal: no recalcula nada)."""
    global _modelo_embeddings, _embeddings_docs, _chunks
    if _modelo_embeddings is None:
        _modelo_embeddings = SentenceTransformer(EMBEDDING_MODEL)
        _embeddings_docs = np.load(RUTA_EMBEDDINGS)
        with open(RUTA_CHUNKS, "rb") as f:
            _chunks = pickle.load(f)


def recuperar(pregunta: str, k: int = 3, umbral_similitud: float = 0.35) -> list[dict]:
    """
    Calcula similitud coseno de 'pregunta' contra TODOS los documentos
    (producto punto sobre vectores normalizados) y devuelve hasta k chunks
    por encima del umbral, ordenados de mayor a menor similitud.
    """
    _cargar_recursos()

    embedding_pregunta = _modelo_embeddings.encode(
        [pregunta], normalize_embeddings=True
    )
    similitudes = np.dot(_embeddings_docs, embedding_pregunta.T).flatten()

    # Top-k por similitud (equivalente a k llamadas a np.argmax sin repetir índices,
    # pero vectorizado): argsort ordena ascendente, [::-1] invierte, [:k] recorta.
    indices_top_k = np.argsort(similitudes)[::-1][:k]

    resultados = []
    for idx in indices_top_k:
        score = float(similitudes[idx])
        if score < umbral_similitud:
            continue
        resultados.append({**_chunks[idx], "score": score})
    return resultados


def construir_contexto_para_prompt(chunks_recuperados: list[dict]) -> str:
    """Concatena los chunks recuperados en un bloque de contexto para el LLM."""
    if not chunks_recuperados:
        return "No se encontró información relevante en la base de conocimiento."
    bloques = [c["texto_contexto"] for c in chunks_recuperados]
    return "\n\n---\n\n".join(bloques)

construir_y_guardar_indice(archivo_origen)

pregunta_prueba = "El auto tiene la luz ABS encendida y el pedal de freno pulsa"
resultados = recuperar(pregunta_prueba, k=3)
for r in resultados:
    print(f"[score={r['score']:.3f}] {r['metadata']['category']} / {r['metadata']['subcategory']}")
print("\n--- Contexto ensamblado ---")
print(construir_contexto_para_prompt(resultados))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddings generados: (99, 384)
Embeddings guardados en /content/rag_embeddings.npy, chunks en /content/rag_chunks.pkl
[score=0.792] Sistema ABS / Módulo de control ABS
[score=0.740] Sistemas líquidos / líquido de frenos
[score=0.669] Tren de transmisión / Diferencial

--- Contexto ensamblado ---
Categoría: Sistema ABS
Subcategoría: Módulo de control ABS
Síntomas asociados: Luz de advertencia del ABS encendida, Pulsación del pedal de freno
Pasos de diagnóstico:
  1. Comprobar el fusible del ABS (Posibles resultados: Fundido / Intacto)
  2. Inspeccionar el cableado del módulo ABS (Posibles resultados: Cableado defectuoso / Cableado correcto)

---

Categoría: Sistemas líquidos
Subcategoría: líquido de frenos
Síntomas asociados: Pedal de freno blando, Luz de advertencia de freno encendida
Pasos de diagnóstico:
  1. Comprobar el nivel y el estado del líquido de frenos (Posibles resultados: Líquido bajo/sucio / Líquido limpio/suficiente)
  2. Inspeccionar las líneas de freno en busca de fug

# 2. Preparación del modelo

## 2.1 Carga del Modelo base

In [6]:
# Cargar el modelo base de Llama y su tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "meta-llama/Llama-3.2-3B-Instruct"
# variante oficial de Meta "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Modelo base cargado: meta-llama/Llama-3.2-3B-Instruct


In [7]:
print("Tipo de 'modelo':", type(modelo))
print("¿'modelo' ya es un PeftModel? ->", hasattr(modelo, "peft_config"))

Tipo de 'modelo': <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
¿'modelo' ya es un PeftModel? -> False


In [8]:
def generar_respuesta_rag(modelo_a_usar, pregunta: str, k: int = 1, max_new_tokens: int = 300):
    chunks = recuperar(pregunta, k=k)
    contexto = construir_contexto_para_prompt(chunks)

    mensajes = [
        {"role": "system", "content": SYSTEM_PROMPT_RAG},
        {"role": "user", "content": f"Contexto recuperado:\n{contexto}\n\nPregunta: {pregunta}"},
    ]
    entrada = tokenizer.apply_chat_template(
        mensajes, add_generation_prompt=True, return_tensors="pt"
    ).to(modelo_a_usar.device)

    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    respuesta = tokenizer.decode(tokens_nuevos, skip_special_tokens=True).strip()

    return {
        "respuesta": respuesta,
        "chunks_recuperados": [
            {"categoria": c["metadata"]["category"], "subcategoria": c["metadata"]["subcategory"], "score": c["score"]}
            for c in chunks
        ],
    }

prompt_prueba = "Se me presentó un vehículo con Luz de advertencia de freno encendida. ¿Qué puede estar pasando?"

respuesta_base = generar_respuesta_rag(modelo, prompt_prueba)
print("Respuesta de ejemplo:"+respuesta_base['respuesta'])

Respuesta de ejemplo:Con la información proporcionada, hay varias posibilidades para qué puede estar sucediendo con el vehículo que tiene la Luz de Advertencia de Freno encendada. Sin embargo, no tengo suficiente información para determinar con certeza el problema específico.

Sin embargo, puedo decir que la luz de advertencias de frenado suele encenderse cuando el sistema de frenados del vehículo detecta una condición que podría afectar la seguridad del conductor o del pasajero. Algunas de las posibles causas que podrían estar relacionadas con la luz encendidas son:

- Problemas con el líquid de frenas, como un nivel bajo o sucio.
- Fugas en las líneas de frenaje.
- Problema con el sensor de frenada o el sistema electrónico de frenadas.

Para determinar la causa específica, sería necesario seguir con el diagnósticos paso a paso, comenzando por comprobar los niveles y el Estado del líq. de frenes (paso 1) y luego inspeccionando las línea de frenajes (passo 2).


In [9]:
#Cargar el dataset de entrenamiento y validación
from datasets import load_dataset
TRAIN_FILE = ruta_destino+"/train_rag.jsonl"
VAL_FILE = ruta_destino+"/val_rag.jsonl"
TEST_FILE = ruta_destino+"/test.jsonl"
dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE},
)

def formatear_chat(example):
    """Aplica el chat template propio de Llama 3.2 a cada conversación."""
    texto = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": texto}


dataset = dataset.map(formatear_chat, remove_columns=dataset["train"].column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/869 [00:00<?, ? examples/s]

Map:   0%|          | 0/110 [00:00<?, ? examples/s]

## 2.2 Configuración de LoRA

In [10]:
# Configurar LoRA (rango, alpha, módulos objetivo) y aplicarlo al modelo base
!pip uninstall -y torchao --quiet
from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()
# r más alto = más capacidad para aprender, pero también más parámetros entrenables

trainable params: 12,156,928 || all params: 3,224,906,752 || trainable%: 0.3770


## 2.3 Configración del entrenador y hacer el entrenamiento

In [11]:
# Configurar el entrenador (SFTTrainer)

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados",
    num_train_epochs=4,
    save_strategy="epoch",
    per_device_train_batch_size=5,
    learning_rate=1e-4,
    logging_steps=1,
    dataset_text_field="text",
    max_length=512,
    report_to="none",
    fp16=True,
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=config_entrenamiento,
)


Adding EOS to train dataset:   0%|          | 0/869 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/869 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/869 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/869 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/869 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

In [12]:
#Ejecutar el fine-tuning
resultado_entrenamiento = trainer.train()
print("Pérdida final:", resultado_entrenamiento.training_loss)

{'loss': '2.942', 'grad_norm': '1.78', 'learning_rate': '0.0001', 'entropy': '1.726', 'num_tokens': '1236', 'mean_token_accuracy': '0.5118', 'epoch': '0.005747'}
{'loss': '2.699', 'grad_norm': '1.502', 'learning_rate': '9.986e-05', 'entropy': '1.756', 'num_tokens': '2509', 'mean_token_accuracy': '0.5323', 'epoch': '0.01149'}
{'loss': '2.493', 'grad_norm': '1.446', 'learning_rate': '9.971e-05', 'entropy': '1.793', 'num_tokens': '3778', 'mean_token_accuracy': '0.5554', 'epoch': '0.01724'}
{'loss': '2.496', 'grad_norm': '1.43', 'learning_rate': '9.957e-05', 'entropy': '1.882', 'num_tokens': '5011', 'mean_token_accuracy': '0.5358', 'epoch': '0.02299'}
{'loss': '2.29', 'grad_norm': '1.402', 'learning_rate': '9.943e-05', 'entropy': '1.862', 'num_tokens': '6312', 'mean_token_accuracy': '0.591', 'epoch': '0.02874'}
{'loss': '2.099', 'grad_norm': '1.292', 'learning_rate': '9.928e-05', 'entropy': '1.863', 'num_tokens': '7551', 'mean_token_accuracy': '0.6037', 'epoch': '0.03448'}
{'loss': '2.054'

In [14]:
import os
from google.colab import drive
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir la ruta
ruta_adaptador = "/content/drive/MyDrive/IA_Aplicada_Con_LLAMA/llama32_3B-adapter-diagnostico-lora-rag"

# 3. Validar si el directorio ya existe
if os.path.exists(ruta_adaptador):
    print(f"⚠️ El directorio ya existe: {ruta_adaptador}")
    print("Cargando adaptador...")
    tokenizer = AutoTokenizer.from_pretrained(ruta_adaptador)
    modelo_lora = PeftModel.from_pretrained(modelo, ruta_adaptador)
else:

    # 4. Crear el directorio y guardar si no existe
    os.makedirs(ruta_adaptador, exist_ok=True)
    trainer.save_model(ruta_adaptador)
    tokenizer.save_pretrained(ruta_adaptador)
    print(f"✅ Modelo y tokenizador guardados exitosamente en: {ruta_adaptador}")

modelo_lora.eval()
print(generar_respuesta_rag(modelo_lora, prompt_prueba))

Mounted at /content/drive
✅ Modelo y tokenizador guardados exitosamente en: /content/drive/MyDrive/IA_Aplicada_Con_LLAMA/llama32_3B-adapter-diagnostico-lora-rag
{'respuesta': 'Est', 'chunks_recuperados': [{'categoria': 'Sistemas líquidos', 'subcategoria': 'líquido de frenos', 'score': 0.7389258146286011}]}


## 2.4 Comparación de perdida (loss)

In [15]:
# Comparar la pérdidas
perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = resultado_entrenamiento.training_loss

print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

# trainer.state.log_history[0]['loss'] es la pérdida después del primer paso registrado,
# no la pérdida real del modelo sin ningún entrenamiento.

Pérdida al inicio del entrenamiento: 2.94
Pérdida final del entrenamiento: 0.15
Reducción: 95%


## 2.5 Validar ejemplo después del enternamiento

In [16]:
# Prueba A: quitar no_repeat_ngram_size por completo (greedy puro)
def generar_respuesta_rag(modelo_a_usar, pregunta, k=2, umbral=0.5, max_new_tokens=300):
    chunks = recuperar(pregunta, k=k, umbral_similitud=umbral)
    if not chunks:
        return {"respuesta": "No tengo información suficiente.", "chunks_recuperados": []}
    contexto = construir_contexto_para_prompt(chunks)

    mensajes = [
        {"role": "system", "content": SYSTEM_PROMPT_RAG},
        {"role": "user", "content": f"Con base en esta información: {contexto}\n\nResponde a: {pregunta}"},
    ]
    entrada = tokenizer.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt").to(modelo_a_usar.device)

    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        # no_repeat_ngram_size ELIMINADO
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    respuesta = tokenizer.decode(tokens_nuevos, skip_special_tokens=True).strip()
    return {"respuesta": respuesta, "chunks_recuperados": [
        {"categoria": c["metadata"]["category"], "subcategoria": c["metadata"]["subcategory"], "score": c["score"]}
        for c in chunks
    ]}

In [17]:
respuesta_ajustada = generar_respuesta_rag(modelo_lora, prompt_prueba)
print("\nRespuesta del modelo ajustado (referencia):\n", respuesta_ajustada)


Respuesta del modelo ajustado (referencia):
 {'respuesta': 'Est', 'chunks_recuperados': [{'categoria': 'Sistemas líquidos', 'subcategoria': 'líquido de frenos', 'score': 0.7389258146286011}, {'categoria': 'Tren de transmisión', 'subcategoria': 'Diferencial central', 'score': 0.7224656939506531}]}


In [18]:
resultados = recuperar(
    "Se me presentó un vehículo con Luz de advertencia de freno encendida. ¿Qué puede estar pasando?",
    k=8
)
for r in resultados:
    print(f"[score={r['score']:.3f}] {r['metadata']['category']} / {r['metadata']['subcategory']}")

[score=0.739] Sistemas líquidos / líquido de frenos
[score=0.722] Tren de transmisión / Diferencial central
[score=0.709] Sistema ABS / Módulo de control ABS
[score=0.673] Tren de transmisión / Acoplamiento viscoso
[score=0.669] Sistema de emisiones / Sensor de presión del tanque de combustible
[score=0.662] Tren de transmisión / Diferencial
[score=0.657] Sistema de emisiones / Válvula de purga del depósito
[score=0.657] Sistema de emisiones / Sensor MAP


In [19]:
resultados = recuperar(
    "¿Qué significa que un auto tenga Olor a humedad procedente de las rejillas de ventilación?",
    k=2
)
for r in resultados:
    print(f"[score={r['score']:.3f}] {r['metadata']['category']} / {r['metadata']['subcategory']}")

[score=0.807] Sistema de refrigeración / Manguera del calentador
[score=0.801] Sistema de aire acondicionado / Evaporador de aire acondicionado


In [23]:
import json
TEST_FILE = ruta_destino+"/test.jsonl"
with open(TEST_FILE, encoding="utf-8") as f:
    ejemplo_test = json.loads(f.readline())  # o cualquier línea de test.jsonl

pregunta_test = next(m["content"] for m in ejemplo_test["messages"] if m["role"] == "user")
print("Pregunta:", pregunta_test)

resultado = generar_respuesta_rag(modelo_lora, pregunta_test, k=3, umbral=0.5)
print(resultado)

Pregunta: Estoy revisando un carro y tiene Pedal de freno blando; luz de advertencia de freno encendida. ¿Por dónde empiezo a diagnosticar?
{'respuesta': 'Esto corresponde al sistema **Sistemas líquidos**, específicamente **líquido de frenos**.\n\nPasos de diagnóstico recomendados:\n1. Comprobar el nivel y el estado del líquido de frenos\n   - Posibles resultados: Líquido bajo/sucio / Líquido limpio/suficiente\n2. Inspeccionar las líneas de freno en busca de fugas\n   - Posibles resultados: Líneas con fugas / Líneas en buen estado', 'chunks_recuperados': [{'categoria': 'Sistemas líquidos', 'subcategoria': 'líquido de frenos', 'score': 0.7477949261665344}, {'categoria': 'Sistema ABS', 'subcategoria': 'Módulo de control ABS', 'score': 0.718209445476532}, {'categoria': 'Tren de transmisión', 'subcategoria': 'Eje y neumáticos', 'score': 0.6331049203872681}]}


# 3 Validación

In [30]:
"""
Motor GENÉRICO de evaluación para tareas de generación de texto estructurado.

No conoce nada sobre "diagnóstico automotriz" ni ningún otro dominio -- solo
sabe comparar dos textos (referencia vs. generado) según CAMPOS que tú
defines con tus propias funciones de extracción (regex, parsers, lo que sea).

Uso típico:
    1. Escribes funciones "extractor" que dado un texto devuelven:
       - un string (o None) para campos tipo "texto" (ej. categoría de un
         ticket, veredicto de un contrato, diagnóstico de un caso clínico)
       - una list[str] para campos tipo "lista" (ej. pasos a seguir, items
         de una factura, cláusulas mencionadas)
    2. Declaras esos campos con DefinicionCampo(...)
    3. Llamas a comparar_respuestas(referencia, generada, tus_campos)
    4. Agregas muchos resultados con promediar_resultados(...)

Este archivo NO importa nada de tu dominio -- eso vive en un "perfil"
aparte (ver ejemplo: perfil_evaluacion_automotriz.py).
"""

import re
import difflib
from dataclasses import dataclass, field
from typing import Callable, Literal, Any

from rouge_score import rouge_scorer

_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

TipoCampo = Literal["texto", "lista"]


# --- Definición de campos (lo que el usuario configura por proyecto) ---

@dataclass
class DefinicionCampo:
    nombre: str
    tipo: TipoCampo
    extractor: Callable[[str], Any]  # "texto" -> str|None ; "lista" -> list[str]
    umbral_similitud: float = 0.85       # usado si tipo == "texto"
    umbral_similitud_items: float = 0.6  # usado si tipo == "lista" (matching difuso por item)


# --- Resultados de comparación ---

@dataclass
class ResultadoCampoTexto:
    nombre: str
    correcto: bool
    valor_referencia: str | None
    valor_generado: str | None


@dataclass
class ResultadoCampoLista:
    nombre: str
    recall: float
    precision: float


@dataclass
class ResultadoComparacion:
    campos: dict[str, ResultadoCampoTexto | ResultadoCampoLista]
    rouge_l_f1: float
    respuesta_vacia_o_ilegible: bool


# --- Utilidades de matching difuso (compartidas por cualquier dominio) ---

def _similar(a: str, b: str, umbral: float) -> bool:
    """Matching difuso de texto corto -- tolera errores de ortografía/fraseo menores."""
    return difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio() >= umbral


def _recall_precision_conjuntos(referencia: list[str], generado: list[str], umbral: float) -> tuple[float, float]:
    """Recall/precision con matching difuso (no exact match) sobre dos listas de strings."""
    if not referencia and not generado:
        return 1.0, 1.0
    if not referencia:
        return 1.0, 0.0
    if not generado:
        return 0.0, 0.0

    ref_cubiertos = 0
    gen_usados = set()
    for r in referencia:
        for i, g in enumerate(generado):
            if i not in gen_usados and _similar(r, g, umbral):
                ref_cubiertos += 1
                gen_usados.add(i)
                break

    recall = ref_cubiertos / len(referencia)
    precision = len(gen_usados) / len(generado)
    return recall, precision


# --- Función principal: comparar UN par (referencia, generada) ---

def comparar_respuestas(
    referencia: str, generada: str, definicion_campos: list[DefinicionCampo]
) -> ResultadoComparacion:
    campos: dict[str, ResultadoCampoTexto | ResultadoCampoLista] = {}
    todo_vacio_en_generada = True

    for campo in definicion_campos:
        val_ref = campo.extractor(referencia)
        val_gen = campo.extractor(generada)

        if campo.tipo == "texto":
            hay_valor_gen = val_gen is not None and val_gen != ""
            if hay_valor_gen:
                todo_vacio_en_generada = False
            correcto = hay_valor_gen and _similar(val_ref or "", val_gen, campo.umbral_similitud)
            campos[campo.nombre] = ResultadoCampoTexto(
                nombre=campo.nombre, correcto=correcto,
                valor_referencia=val_ref, valor_generado=val_gen,
            )
        else:  # "lista"
            val_ref = val_ref or []
            val_gen = val_gen or []
            if val_gen:
                todo_vacio_en_generada = False
            recall, precision = _recall_precision_conjuntos(val_ref, val_gen, campo.umbral_similitud_items)
            campos[campo.nombre] = ResultadoCampoLista(
                nombre=campo.nombre, recall=recall, precision=precision,
            )

    rouge_l = _rouge.score(referencia, generada)["rougeL"].fmeasure

    return ResultadoComparacion(
        campos=campos,
        rouge_l_f1=rouge_l,
        respuesta_vacia_o_ilegible=todo_vacio_en_generada,
    )


def promediar_resultados(resultados: list[ResultadoComparacion]) -> dict[str, float]:
    """Agrega una lista de ResultadoComparacion en un dict plano de métricas
    promedio, sin importar qué campos se hayan definido (genérico)."""
    n = len(resultados)
    if n == 0:
        return {}

    metricas: dict[str, float] = {}
    nombres_campos = resultados[0].campos.keys()

    for nombre in nombres_campos:
        primero = resultados[0].campos[nombre]
        if isinstance(primero, ResultadoCampoTexto):
            metricas[f"{nombre}_correcto (%)"] = 100 * sum(
                r.campos[nombre].correcto for r in resultados
            ) / n
        else:
            metricas[f"{nombre}_recall"] = sum(r.campos[nombre].recall for r in resultados) / n
            metricas[f"{nombre}_precision"] = sum(r.campos[nombre].precision for r in resultados) / n

    metricas["rouge_l_f1"] = sum(r.rouge_l_f1 for r in resultados) / n
    metricas["respuestas_ilegibles (%)"] = 100 * sum(
        r.respuesta_vacia_o_ilegible for r in resultados
    ) / n
    return metricas


# --- Fábricas de extractores genéricos (reutilizables en cualquier proyecto) ---

def extractor_regex_grupo(patron: str, grupo: int = 1, flags=re.IGNORECASE) -> Callable[[str], str | None]:
    """Crea un extractor tipo 'texto': busca 'patron' y devuelve el grupo
    capturado, o None si no matchea. Sirve para cualquier campo escalar
    (categoría, veredicto, clasificación, nivel de urgencia, etc.)."""
    regex = re.compile(patron, flags)

    def extractor(texto: str) -> str | None:
        m = regex.search(texto)
        return m.group(grupo).strip() if m else None

    return extractor


def extractor_lista_lineas(patron_item: str, grupo: int = 1, flags=re.MULTILINE) -> Callable[[str], list[str]]:
    """Crea un extractor tipo 'lista': cada línea que matchee 'patron_item'
    aporta un elemento a la lista. Sirve para listas simples de un solo nivel
    (pasos, items de una factura, cláusulas, etc.)."""
    regex = re.compile(patron_item, flags)

    def extractor(texto: str) -> list[str]:
        return [m.group(grupo).strip() for m in regex.finditer(texto)]

    return extractor


def extractores_dos_niveles(
    patron_item: str, patron_subitem: str, grupo_item: int = 1, grupo_subitem: int = 1
) -> tuple[Callable[[str], list[str]], Callable[[str], list[str]]]:
    """
    Para estructuras de dos niveles (ej. 'pasos', cada uno con sus propios
    'resultados posibles' en la línea siguiente). Devuelve DOS extractores:
    uno para los items de primer nivel, otro para todos los subitems
    (aplanados) asociados a cualquier item.

    Uso: extractor_pasos, extractor_resultados = extractores_dos_niveles(...)
    """
    regex_item = re.compile(patron_item)
    regex_subitem = re.compile(patron_subitem, re.IGNORECASE)

    def _parsear(texto: str) -> list[dict]:
        items = []
        item_actual = None
        for linea in texto.splitlines():
            m_item = regex_item.match(linea)
            if m_item:
                item_actual = {"item": m_item.group(grupo_item).strip(), "subitems": []}
                items.append(item_actual)
                continue
            m_sub = regex_subitem.search(linea)
            if m_sub and item_actual is not None:
                partes = [p.strip() for p in m_sub.group(grupo_subitem).split("/")]
                item_actual["subitems"].extend(partes)
        return items

    def extractor_items(texto: str) -> list[str]:
        return [d["item"] for d in _parsear(texto)]

    def extractor_subitems_aplanados(texto: str) -> list[str]:
        return [s for d in _parsear(texto) for s in d["subitems"]]

    return extractor_items, extractor_subitems_aplanados

In [32]:
"""
Configuración específica del proyecto de diagnóstico automotriz, usando el
motor GENÉRICO de metricas_evaluacion.py.

Esto es lo único que tendrías que reescribir si adaptas la evaluación a otro
proyecto (ej. clasificación de tickets de soporte, resúmenes de contratos,
notas clínicas) -- el motor (metricas_evaluacion.py) no cambia nunca.
"""
"""
from metricas_evaluacion import (
    DefinicionCampo,
    extractor_regex_grupo,
    extractores_dos_niveles,
)
"""
# Campos escalares: categoría y subcategoría, extraídas de
# "Esto corresponde al sistema **X**, específicamente **Y**."
extractor_categoria = extractor_regex_grupo(
    r"\*\*(.+?)\*\*,\s*específicamente\s*\*\*.+?\*\*", grupo=1
)
extractor_subcategoria = extractor_regex_grupo(
    r"\*\*.+?\*\*,\s*específicamente\s*\*\*(.+?)\*\*", grupo=1
)

# Campos de lista de dos niveles: pasos numerados, cada uno con sus
# "Posibles resultados: A / B" (tolerante a errores ortográficos del modelo
# en la palabra "resultados", ver resultad\w*).
extractor_pasos, extractor_resultados = extractores_dos_niveles(
    patron_item=r"^\s*\d+\.\s*(.+)$",
    patron_subitem=r"(?:posibles?\s+)?resultad\w*:?\s*(.+)$",
)

CAMPOS_AUTOMOTRIZ = [
    DefinicionCampo("categoria", "texto", extractor_categoria, umbral_similitud=0.85),
    DefinicionCampo("subcategoria", "texto", extractor_subcategoria, umbral_similitud=0.85),
    DefinicionCampo("pasos", "lista", extractor_pasos, umbral_similitud_items=0.6),
    DefinicionCampo("resultados", "lista", extractor_resultados, umbral_similitud_items=0.6),
]

In [1]:
# =============================================================================
# Evaluación COMPLETA del pipeline RAG sobre test.jsonl.
# Corre esto en la misma sesión de Colab, con modelo_lora, tokenizer,
# generar_respuesta, generar_respuesta_rag_sin_ngram, recuperar (retriever_rag)
# ya definidos/importados.
#
# Requiere: metricas_evaluacion.py y perfil_evaluacion_automotriz.py subidos
# a la sesión (o pegados en celdas).
# =============================================================================

import json
'''
from metricas_evaluacion import comparar_respuestas, promediar_resultados
from perfil_evaluacion_automotriz import (
    CAMPOS_AUTOMOTRIZ, extractor_categoria, extractor_subcategoria,
)
from retriever_rag import recuperar
'''

TEST_FILE = ruta_destino + "/test.jsonl"

test_examples = []
with open(TEST_FILE, encoding="utf-8") as f:
    for linea in f:
        ejemplo = json.loads(linea)
        user_msg = next(m["content"] for m in ejemplo["messages"] if m["role"] == "user")
        respuesta_ref = next(m["content"] for m in ejemplo["messages"] if m["role"] == "assistant")
        test_examples.append({"user": user_msg, "referencia": respuesta_ref})

print(f"Ejemplos de test cargados: {len(test_examples)}")

# Limitar a una muestra (para correr evaluaciones rápidas mientras
# iteras). Semilla fija para que la muestra sea la misma entre corridas y
# los resultados sean comparables. Sube N_MUESTRAS (o quítalo) para la
# evaluación final completa.
import random

N_MUESTRAS = 100
if N_MUESTRAS is not None and N_MUESTRAS < len(test_examples):
    test_examples = random.Random(42).sample(test_examples, N_MUESTRAS)
    print(f"Usando una muestra de {len(test_examples)} ejemplos (semilla=42)")


# Recall@k del retriever: ¿el documento correcto aparece entre los
# top-k recuperados? (compara metadata del chunk contra categoría/subcategoría
# extraídas de la respuesta de referencia -- reutiliza los extractores del perfil)
def calcular_recall_retriever(test_examples: list[dict], valores_k: list[int]) -> dict[int, float]:
    max_k = max(valores_k)
    aciertos_por_k = {k: 0 for k in valores_k}

    for ej in test_examples:
        cat_ref = extractor_categoria(ej["referencia"])
        subcat_ref = extractor_subcategoria(ej["referencia"])
        recuperados = recuperar(ej["user"], k=max_k, umbral_similitud=0.0)  # umbral 0: no filtrar para medir recall real

        for k in valores_k:
            top_k = recuperados[:k]
            acierto = any(
                c["metadata"]["category"] == cat_ref and c["metadata"]["subcategory"] == subcat_ref
                for c in top_k
            )
            if acierto:
                aciertos_por_k[k] += 1

    return {k: aciertos_por_k[k] / len(test_examples) for k in valores_k}


print("\n=== Recall@k del retriever ===")
recall_k = calcular_recall_retriever(test_examples, valores_k=[1, 3, 5])
for k, r in recall_k.items():
    print(f"Recall@{k}: {r*100:.1f}%")

# Interpretación: si Recall@3 es bajo, ningún ajuste al modelo generador va a
# compensarlo -- el documento correcto ni siquiera llega a estar disponible
# en el contexto. Este número pone un techo superior a la calidad posible
# del sistema completo.


def condicion_finetuned_con_rag(pregunta: str) -> str:
    resultado = generar_respuesta_rag(modelo_lora, pregunta, k=3, umbral=0.5)
    return resultado["respuesta"]


condiciones = {
    "fine-tuned + RAG": condicion_finetuned_con_rag,
  }

modelo_lora.eval()

resultados_por_condicion: dict[str, list] = {nombre: [] for nombre in condiciones}

for i, ej in enumerate(test_examples):
    for nombre, funcion in condiciones.items():
        respuesta = funcion(ej["user"])
        resultados_por_condicion[nombre].append(
            comparar_respuestas(ej["referencia"], respuesta, CAMPOS_AUTOMOTRIZ)
        )
    print(f"[{i+1}/{len(test_examples)}] procesado")


#Tabla comparativa final
metricas_por_condicion = {
    nombre: promediar_resultados(resultados)
    for nombre, resultados in resultados_por_condicion.items()
}

nombres_metricas = list(next(iter(metricas_por_condicion.values())).keys())
anchos_columna = 18

encabezado = f"{'Métrica':<30}" + "".join(f"{c:>{anchos_columna}}" for c in condiciones)
print("\n" + encabezado)
print("-" * len(encabezado))
for m in nombres_metricas:
    fila = f"{m:<30}" + "".join(
        f"{metricas_por_condicion[c][m]:>{anchos_columna}.2f}" for c in condiciones
    )
    print(fila)

print(f"\nRecall@3 del retriever (referencia): {recall_k[3]*100:.1f}%")

NameError: name 'ruta_destino' is not defined

In [38]:
recall_crudo = calcular_recall_retriever(test_examples, valores_k=[3])[3]


def calcular_recall_con_umbral(test_examples, k=3, umbral=0.5):
    aciertos = 0
    for ej in test_examples:
        cat_ref = extractor_categoria(ej["referencia"])
        subcat_ref = extractor_subcategoria(ej["referencia"])
        recuperados = recuperar(ej["user"], k=k, umbral_similitud=umbral)  # <-- umbral real de producción
        if any(c["metadata"]["category"] == cat_ref and c["metadata"]["subcategory"] == subcat_ref for c in recuperados):
            aciertos += 1
    return aciertos / len(test_examples)


recall_efectivo = calcular_recall_con_umbral(test_examples, k=3, umbral=0.5)

print(f"Recall@3 crudo (umbral=0.0):     {recall_crudo*100:.1f}%")
print(f"Recall@3 efectivo (umbral=0.5):  {recall_efectivo*100:.1f}%")
print(f"Diferencia (documentos correctos descartados por el umbral): {(recall_crudo - recall_efectivo)*100:.1f} puntos")

Recall@3 crudo (umbral=0.0):     70.0%
Recall@3 efectivo (umbral=0.5):  70.0%
Diferencia (documentos correctos descartados por el umbral): 0.0 puntos
